In [ ]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Update data_path to reflect the mounted Google Drive path
data_path = "/content/drive/My Drive/STAT390 Data/All Calls by Month/"

In [ ]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

### Reading first 5 rows of all data files
The code chunk below reads the first 5 rows of all data files. This is to check the columns that are present in all the data files.

In [ ]:
i=0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows = 5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows = 5))
    #df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df[i].shape)
    i = i + 1

0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 64)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)
17 September 2025 (5, 69)


The code chunk below identifies the columns missing in at least one DataFrame.

In [ ]:
all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))
common_cols
not_in_all = all_cols - common_cols
print("Columns missing from at least one dataframe:", not_in_all)

Columns missing from at least one dataframe: {'User', 'Answered Elsewhere', 'Hold Duration', 'External caller ID number', 'Queue Type', 'Device owner UUID', 'PSTN vendor name2', 'Auto Attendant Key Pressed', 'Public Called IP Address', 'Redirecting party UUID', 'Recall Type', 'Original called party UUID', 'Call Recording Result', 'Call Recording Trigger', 'Call Recording Platform Name', 'Public Calling IP Address', 'Column1'}


The code chunk below prints the columns present in all the data files.

In [ ]:
print(common_cols)

{'Outbound trunk', 'Local SessionID', 'Org UUID', 'Local call ID', 'Direction', 'Inbound trunk', 'Client type', 'Correlation ID', 'Original reason', 'Transfer related call ID', 'Sub client type', 'PSTN legal entity', 'Related call ID', 'PSTN vendor name', 'Client version', 'Final local sessionID', 'Report time', 'Call outcome reason', 'OS type', 'Call type', 'Network call ID', 'PSTN vendor Org ID', 'Route group', 'User UUID', 'Location', 'Call transfer time', 'Authorization code', 'PSTN provider ID', 'Releasing party', 'User number', 'Call outcome', 'Model', 'Redirect reason', 'Call ID', 'Remote SessionID', 'Final remote sessionID', 'Answered', 'Device Mac', 'Called number', 'Start time', 'Duration', 'Site main number', 'Site timezone', 'Report ID', 'Redirecting number', 'Remote call ID', 'Ring duration', 'Answer Indicator', 'Release time', 'International Country', 'Related reason', 'Site UUID', 'User type', 'Answer time', 'Department ID'}


### Reading all the data files
All the datafiles are read with the common columns read first.

In [ ]:
df_main = pd.DataFrame(columns=list(common_cols))

In [ ]:
i=0;
for f in files:
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)
    i = i + 1

0 April 2024 (56662, 63)
1 April 2025 (63636, 63)
2 August 2024 (63262, 57)
3 August 2025 (57071, 63)
4 December 2024 (49445, 55)
5 February 2025 (63669, 63)
6 January 2025 (62623, 64)
7 July 2024 (62292, 55)
8 July 2025 (60438, 63)
9 June 2024 (56763, 63)
10 June 2025 (54598, 63)
11 March 2025 (59149, 64)
12 May 2024 (62944, 63)
13 May 2025 (55428, 63)
14 November 2024 (49953, 63)
15 October 2024 (62354, 55)
16 September 2024 (61250, 55)
17 September 2025 (56571, 69)


### Converting date to datetime format

In [ ]:
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)

In [ ]:
df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

In [ ]:
df_main["Start time"].head()

,Start time
0,2024-04-30 18:58:53.988
1,2024-04-30 18:56:37.386
2,2024-04-30 18:54:59.099
3,2024-04-30 18:54:59.099
4,2024-04-30 18:54:52.336


In [ ]:
df_allcallsdata = df_main.copy()

# New Step

In [ ]:
df_allcallsdata.sort_values(by=['Correlation ID', 'Start time'], ascending=[True, True])[['Correlation ID', 'Start time', 'Called number', 'Duration']].head(50)

,Correlation ID,Start time,Called number,Duration
600224,00000fe9-dfa0-41c5-986b-d9ce731f2715,2025-06-27 11:25:23.791,13123411070,4
929258,00001e73-ce33-48f1-b531-bddc7ee3965d,2024-10-05 12:30:09.903,13124312299,1961
267542,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:41:47.355,13123478311,44
267540,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:42:05.358,13123478300,44
267541,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:42:05.358,13123478300,44
818656,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.746,13122296344,65
818654,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.749,13123478300,65
818655,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.749,13123478300,65
974134,000090ae-a71c-49e9-99d6-fdc7078dfa48,2024-09-13 14:35:32.428,17086568223,57
828444,0000aef0-6f35-4560-a4be-5b3bc31fbefb,2024-11-28 09:49:47.564,13123411070,2


### Determine if a call is outbound or inbound

In [ ]:
### Remove call duration = 0
# convert the "Duration" column into numeric
df_allcallsdata["Duration"] = pd.to_numeric(df_allcallsdata["Duration"], errors="coerce")
df_allcallsdata = df_allcallsdata[df_allcallsdata["Duration"] > 0]


### Sort by Correlation ID and Start Time (chronological order)
df_allcallsdata = df_allcallsdata.sort_values(by=["Correlation ID", "Start time"])


### Classify call type (Inbound/Outbound/Internal)
def classify_call(row):
    if row["PSTN vendor name"] == "CallTower" and row["Direction"] == "ORIGINATING":
        return "Outbound"
    elif row["PSTN vendor name"] == "CallTower" and row["Direction"] == "TERMINATING":
        return "Inbound"
    elif row["PSTN vendor name"] == "NA":
        return "Internal"
    else:
        return "Other"

# Temporary column for per-row classification
df_allcallsdata["TempCallType"] = df_allcallsdata.apply(classify_call, axis=1)

# Propagate earliest-leg call type to all legs of the same call
earliest_calltype = df_allcallsdata.groupby("Correlation ID").first().reset_index()[["Correlation ID", "TempCallType"]]
df_allcallsdata = df_allcallsdata.drop(columns=["TempCallType"])
df_allcallsdata = df_allcallsdata.merge(earliest_calltype.rename(columns={"TempCallType":"Inbound/Outbound"}),
                                        on="Correlation ID", how="left")

In [ ]:
# Step 3: Extract time features
df_allcallsdata["Start time"] = pd.to_datetime(df_allcallsdata["Start time"])
df_allcallsdata["Hour"] = df_allcallsdata["Start time"].dt.hour
df_allcallsdata["DayOfWeek"] = df_allcallsdata["Start time"].dt.weekday + 1  # Monday=1, Sunday=7
df_allcallsdata["Month"] = df_allcallsdata["Start time"].dt.month
df_allcallsdata["Quarter"] = df_allcallsdata["Start time"].dt.quarter
df_allcallsdata["Year"] = df_allcallsdata["Start time"].dt.year

In [ ]:
# Step 4: Filtering by service description

service_numbers = ['13123478347', '18882652188', '13124235904', '13122296344', '13124235900',
                   '13122296071', '13123478392', '13124235909', '13123478309', '13122296072',
                   '13124235938', '13124312101', '13123478340', '13122296073', '18004459025',
                   '18884018200', '13122296014', '13123411070', '13122296300', '13125068646',
                   '13124312299', '13122296079', '13125068647']

### df_allcallsdata_number_filtered = df_allcallsdata[df_allcallsdata["Called number"].isin(service_numbers) | df_allcallsdata["User number"].isin(service_numbers)]

In [ ]:
# Explore the frequency of the services numbers in all number-related columns

print("Number of explained number as 'Site main number':", df_allcallsdata.loc[df_allcallsdata['Site main number'].isin(service_numbers), :].shape)
print("Number of explained number as 'User number':", df_allcallsdata.loc[df_allcallsdata['User number'].isin(service_numbers), :].shape)
print("Number of explained number as 'External caller ID main number':", df_allcallsdata.loc[df_allcallsdata['External caller ID number'].isin(service_numbers), :].shape)
print("Number of explained number as 'Redirecting number':", df_allcallsdata.loc[df_allcallsdata['Redirecting number'].isin(service_numbers), :].shape)

# Question
## What is external caller ID main number?

Number of explained number as 'Site main number': (0, 78)
Number of explained number as 'User number': (493548, 78)
Number of explained number as 'External caller ID main number': (375544, 78)
Number of explained number as 'Redirecting number': (348589, 78)


In [ ]:
# Explore the frequency of the services numbers in "User number" and "Called number" for inbound/outbound calls

print("Number of explained number as 'User number' in inbound calls:", df_allcallsdata.loc[(df_allcallsdata['User number'].isin(service_numbers)) & (df_allcallsdata['Inbound/Outbound'] == 'Inbound'), :].shape)
print("Number of explained number as 'Called number' in inbound calls:", df_allcallsdata.loc[(df_allcallsdata['Called number'].isin(service_numbers)) & (df_allcallsdata['Inbound/Outbound'] == 'Inbound'), :].shape)
print("Number of explained number as 'User number' in outbound calls:", df_allcallsdata.loc[(df_allcallsdata['User number'].isin(service_numbers)) & (df_allcallsdata['Inbound/Outbound'] == 'Outbound'), :].shape)
print("Number of explained number as 'Called number' in outbound calls:", df_allcallsdata.loc[(df_allcallsdata['Called number'].isin(service_numbers)) & (df_allcallsdata['Inbound/Outbound'] == 'Outbound'), :].shape)
print("Shape of the dataset:", df_allcallsdata.shape)

# Question
## If a call is outbound, why the LegalAid phone numbers still exist in the user number and called number?
## What's the difference between user number and called number? Why the frequency of phone numbers in 'user number' is larger than that of 'called number'?

Number of explained number as 'User number' in inbound calls: (441104, 78)
Number of explained number as 'Called number' in inbound calls: (351242, 78)
Number of explained number as 'User number' in outbound calls: (1742, 78)
Number of explained number as 'Called number' in outbound calls: (112, 78)
Shape of the dataset: (1003089, 78)


In [ ]:
df_allcallsdata_filtered = df_allcallsdata[
    ((df_allcallsdata['Inbound/Outbound'] == 'Inbound') & df_allcallsdata["Called number"].isin(service_numbers))
    |
    ((df_allcallsdata['Inbound/Outbound'] == 'Outbound') & df_allcallsdata["User number"].isin(service_numbers))
]

In [ ]:
df_allcallsdata_filtered.loc[df_allcallsdata_filtered['Inbound/Outbound'] == 'Inbound']['Called number'].nunique()

20

In [ ]:
df_allcallsdata_filtered.loc[df_allcallsdata_filtered['Inbound/Outbound'] == 'Inbound']['Called number'].value_counts()

,count
Called number,
13123411070,211793
13124235938,52411
13122296300,34026
13125068646,10324
13124312299,9728
13122296344,6882
13122296079,6542
13122296071,6432
13125068647,3716


In [ ]:
df_allcallsdata_filtered.loc[df_allcallsdata_filtered['Inbound/Outbound'] == 'Outbound']['User number'].nunique()

6

In [ ]:
df_allcallsdata_filtered.loc[df_allcallsdata_filtered['Inbound/Outbound'] == 'Outbound']['User number'].value_counts()

,count
User number,
13122296014,1136
13122296079,305
13122296300,234
13125068646,42
13124235938,16
13125068647,9


In [ ]:
# Investigate the porportion of the data remained
print("The proportion of the original dataset remained:", df_allcallsdata_filtered.shape[0] / df_main.shape[0])

The proportion of the original dataset remained: 0.33359921671511794


In [ ]:
# Step 6: Save results
print(df_allcallsdata_filtered.head(20))

df_allcallsdata_filtered.to_csv("df_allcallsdata_filtered2.csv", index=False)

                Outbound trunk                   Local SessionID  \
0   wcc_Pc_tp-ipRwm_ku064NHZiw  a92ac4cc00804e3bb9cbe97b7942d539   
1   wcc_Iyq3fhu9TjS8hbku8c4Zcg  141f435b00804e02be5e1449225771cc   
5                          NaN  8cb760aa4845469eb3f2fb83a3c027a9   
9   wcc_Pc_tp-ipRwm_ku064NHZiw  e2d8c14500804335890fffc0c97b0587   
15  wcc_Pc_tp-ipRwm_ku064NHZiw  eeffce7800804cffb9d52cd377572f5b   
19  wcc_Pc_tp-ipRwm_ku064NHZiw  ca983333008049eaa1950999bbd5477c   
20                         NaN  29f95dc100804481a8cf2150b5aa9a33   
21                         NaN  b3aab2ea335749069f4b48fb6964a832   
28  wcc_Pc_tp-ipRwm_ku064NHZiw  2f796e5600804cfb8533d185b75509ac   
29  wcc_Pc_tp-ipRwm_ku064NHZiw  da089532008043d0b24c0ea305a077d1   
35                         NaN  bd2d78c0825c4948bbb6db48799a6798   
38  wcc_Pc_tp-ipRwm_ku064NHZiw  c448a9630080498cb054fa8a19f9e5f5   
46                         NaN  e64d315bcaf142daab303e8759e81e96   
49                         NaN  0591086f00105000